# 08 – Risk Controls + Live-Readiness Assessment

Three sections:
1. Risk manager demo on real data (circuit breakers, VaR, position caps)
2. Full go/no-go readiness checklist
3. What still needs to be built before live capital

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from core.sp500_loader import download_sp500, get_panel, liquid_universe
from core.risk_manager import RiskManager
from backtesting import PortfolioEngine
from backtesting.evaluation import full_evaluation
from strategies.cross_sectional import MultiFactor
plt.rcParams['figure.figsize'] = (14, 5)
download_sp500()
panel = get_panel(liquid_universe(n=150))
strat = MultiFactor(long_short=False)

## 1 – Risk Controls: before vs after

Each layer adds protection. The 5 circuit-breaker fires correspond to real market events:
`Apr 2014, Oct 2014, Aug 2015 (China crash), Jan 2016, Feb 2018 (VIX spike)`.

In [ ]:
rm = RiskManager(
    max_position_wt=0.05,    # 5% max per name
    max_dd_halt=0.15,        # halt at 15% drawdown
    max_daily_loss=0.03,     # halt if day PnL < -3%
    var_limit=0.015,         # 95% 1-day VaR ≤ 1.5%
    max_beta=1.2,            # portfolio beta ≤ 1.2
    dd_resume_threshold=0.05,
)
uncontrolled = PortfolioEngine('ME', cost_bps=10).run(panel, strat.weights, name='No risk ctrl')
controlled   = PortfolioEngine('ME', cost_bps=10, inverse_vol=True, risk_manager=rm).run(panel, strat.weights, name='Full risk suite')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
uncontrolled.equity_curve.plot(ax=axes[0], label='No risk controls', color='tomato')
controlled.equity_curve.plot(ax=axes[0], label='Full risk suite', color='steelblue')
axes[0].set_title('Equity curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

dd_u = uncontrolled.equity_curve / uncontrolled.equity_curve.cummax() - 1
dd_c = controlled.equity_curve / controlled.equity_curve.cummax() - 1
dd_u.plot(ax=axes[1], label='No risk controls', color='tomato', alpha=0.7)
dd_c.plot(ax=axes[1], label='Full risk suite', color='steelblue')
axes[1].axhline(-0.15, ls='--', color='black', alpha=0.5, label='15% halt threshold')
axes[1].set_title('Drawdown'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Circuit breaker events fired:')
for e in controlled.halt_log: print(' ', e)

In [ ]:
comparison = pd.DataFrame({'No risk controls': full_evaluation(uncontrolled, n_trials=10),
                           'Full risk suite':  full_evaluation(controlled,   n_trials=10)}).T
comparison[['ann_return','sharpe_ratio','max_dd','t_stat','psr_vs_0','deflated_sharpe']].round(3)

## 2 – Go / No-Go Readiness Checklist

Run the cell below for an interactive assessment.

In [ ]:
checks = [
    # (category, item, status, notes)
    # ── STRATEGY ────────────────────────────────────────────────────────
    ('Strategy', 'Economic rationale for signals',       'PASS',   'Momentum (underreaction), low-vol anomaly, reversal — all documented in literature'),
    ('Strategy', 'Out-of-sample backtest',               'PARTIAL','Walk-forward done; no separate holdout period'),
    ('Strategy', 'IC > 0 on real data',                  'PARTIAL','IC ~0.012 — positive but below 0.02 threshold for "good" signal'),
    ('Strategy', 'Deflated Sharpe > 0.9',                'PARTIAL','DSR ~0.5 — marginal, not strong'),
    ('Strategy', 'Tested across multiple market regimes','FAIL',   'Only 2013-2018 bull market. Missing: 2008-09 GFC, 2020 COVID, 2022 bear'),
    ('Strategy', 'Survivorship-bias-free universe',      'FAIL',   'Current dataset is survivorship-biased. Need point-in-time constituents or ETFs'),
    ('Strategy', 'Purged cross-validation (CPCV)',        'FAIL',   'Only simple walk-forward. López de Prado CPCV not yet implemented'),
    # ── RISK CONTROLS ───────────────────────────────────────────────────
    ('Risk',     'Position caps',                        'PASS',   '5% max per name implemented'),
    ('Risk',     'Drawdown circuit breaker',              'PASS',   '15% halt, 5% resume hysteresis, fires on real events'),
    ('Risk',     'Daily loss limit',                     'PASS',   '3% daily loss limit — fired Aug 2015, Jan 2016, Feb 2018'),
    ('Risk',     'VaR monitoring',                       'PASS',   'Historical 95% 1-day VaR with hard scaling'),
    ('Risk',     'Beta / correlation limit',              'PASS',   'Portfolio beta capped at 1.2'),
    ('Risk',     'Sector concentration limit',           'PASS',   '30% max sector — requires sector_map populated'),
    ('Risk',     'Stress test / scenario analysis',      'FAIL',   'No stress test against 2008 or COVID scenarios'),
    # ── INFRASTRUCTURE ──────────────────────────────────────────────────
    ('Infra',    'IBKR connection wrapper',              'PASS',   'core/ibkr_connection.py implemented'),
    ('Infra',    'Paper trading validated (≥3 months)',   'FAIL',   'Not started. This is MANDATORY before live capital'),
    ('Infra',    'Order management / reconciliation',    'PARTIAL','Basic structure only; no live reconciliation vs broker'),
    ('Infra',    'Monitoring & alerting',                'FAIL',   'No live P&L dashboard, no email/SMS alerts'),
    ('Infra',    'Data pipeline (automated daily)',      'FAIL',   'Manual download only; no scheduled fetch'),
    ('Infra',    'Execution quality tracking',           'FAIL',   'No slippage measurement vs model'),
    # ── COMPLIANCE (Ireland / MiFID II) ─────────────────────────────────
    ('Compliance','IBKR account & suitability',          'PASS',   'Assumed: you have an active IBKR Ireland account'),
    ('Compliance','Trading journal / audit log',         'FAIL',   'No automated trade journal'),
    ('Compliance','Position size vs account size',       'WARN',   'Ensure no single position > ~2-5% of total account at risk'),
    ('Compliance','Tax treatment (CGT in Ireland)',      'WARN',   'CGT at 33% in Ireland. Frequent trading increases complexity — consult a tax advisor'),
]

df = pd.DataFrame(checks, columns=['Category','Check','Status','Notes'])
totals = df['Status'].value_counts()

emoji = {'PASS': '✅', 'PARTIAL': '⚠️', 'FAIL': '❌', 'WARN': '⚠️'}
for cat, grp in df.groupby('Category'):
    print(f'\n── {cat.upper()} ──')
    for _, row in grp.iterrows():
        print(f"  {emoji[row.Status]} [{row.Status:7s}] {row.Check}")
        print(f"             {row.Notes}")

print(f"\n{'='*60}")
print(f"SUMMARY: PASS={totals.get('PASS',0)}  PARTIAL={totals.get('PARTIAL',0)}  FAIL={totals.get('FAIL',0)}  WARN={totals.get('WARN',0)}")
print(f"{'='*60}")
print("VERDICT: NOT READY FOR LIVE CAPITAL")
print("Minimum before going live: fix FAIL items in Strategy + Infra (paper trading).")

## 3 – What still needs to be built (priority order)

See notebook bottom for full discussion. Short version:

| Priority | Item | Effort | Impact |
|----------|------|--------|--------|
| 1 | **Paper trading 3-6 months** | Low (just run it) | Critical |
| 2 | **Bear market data** (2008, 2020, 2022) | Medium | High |
| 3 | **ETF-based universe** (no survivorship bias) | Low | High |
| 4 | **CPCV validation** (López de Prado) | Medium | High |
| 5 | **Automated data pipeline** | Medium | Required for live |
| 6 | **Monitoring & alerting** | Medium | Required for live |
| 7 | **ML ranker** (LightGBM) | High | Marginal alpha |
| 8 | **Stress testing** | Low | Important |
